# Case File: Verified Video Investigation for Smart Cities

Pipeline: **index** video segments with a VLM, **search** with a text query, **verify** each candidate with the VLM, then **export** an evidence package (HTML report, clips, JSON).


## 1. Install

In [16]:
!pip -q install transformers accelerate sentence-transformers opencv-python-headless
!apt-get -qq install -y ffmpeg > /dev/null

## 2. Config and imports

In [17]:
import os, io, json, time, base64, zipfile, subprocess, html, shutil
from pathlib import Path
import cv2
import numpy as np
import torch
from PIL import Image

SEGMENT_SECONDS = 5        # length of each indexed segment
FRAMES_PER_SEGMENT = 3     # frames the VLM sees per segment
MAX_SIDE = 448             # resize frames (keeps the T4 fast)
TOP_K = 8                  # candidates to verify per query
VERIFY_THRESHOLD = 0.60    # minimum P(yes) to count as a verified match

WORKDIR = Path("/content/casefile")
VIDEO_DIR = WORKDIR / "videos"
OUT_DIR = WORKDIR / "out"
for d in (VIDEO_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (will be slow)")

GPU: Tesla T4


## 3. Load the VLM and the text embedder

In [18]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from sentence_transformers import SentenceTransformer

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

tok = processor.tokenizer
YES_IDS = {tok.encode(w, add_special_tokens=False)[0] for w in ["Yes", "yes", "YES"]}
NO_IDS = {tok.encode(w, add_special_tokens=False)[0] for w in ["No", "no", "NO"]}


def vlm(images, prompt, max_new_tokens=120, yes_no=False):
    # Returns text, or (text, p_yes) when yes_no=True.
    content = [{"type": "image"} for _ in images] + [{"type": "text", "text": prompt}]
    messages = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=images, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            return_dict_in_generate=True,
            output_scores=yes_no,
        )
    seq = out.sequences[:, inputs.input_ids.shape[1]:]
    answer = processor.batch_decode(seq, skip_special_tokens=True)[0].strip()
    if not yes_no:
        return answer
    probs = torch.softmax(out.scores[0][0].float(), dim=-1)
    p_yes = sum(probs[i].item() for i in YES_IDS)
    p_no = sum(probs[i].item() for i in NO_IDS)
    return answer, p_yes / (p_yes + p_no + 1e-9)

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 4. Upload videos (each file = one camera)

Upload a few traffic clips. The file name becomes the camera name. If you have Hafnia footage or a public traffic dataset, use that.

In [23]:
from google.colab import files
uploaded = files.upload()
for name, data in uploaded.items():
    (VIDEO_DIR / name).write_bytes(data)
VIDEOS = sorted(p for p in VIDEO_DIR.iterdir() if p.suffix.lower() in {".mp4", ".mov", ".avi", ".mkv"})
print("Cameras:", [v.stem for v in VIDEOS])

Saving mixkit-man-cycling-down-a-street-in-a-city-41878-hd-ready.mp4 to mixkit-man-cycling-down-a-street-in-a-city-41878-hd-ready.mp4
Saving mixkit-manhattan-avenue-with-traffic-4335-hd-ready.mp4 to mixkit-manhattan-avenue-with-traffic-4335-hd-ready.mp4
Saving mixkit-person-walking-on-crosswalk-98-hd-ready.mp4 to mixkit-person-walking-on-crosswalk-98-hd-ready.mp4
Saving mixkit-traffic-light-directing-traffic-4272-hd-ready.mp4 to mixkit-traffic-light-directing-traffic-4272-hd-ready.mp4
Cameras: ['mixkit-man-cycling-down-a-street-in-a-city-41878-hd-ready (1)', 'mixkit-man-cycling-down-a-street-in-a-city-41878-hd-ready', 'mixkit-manhattan-avenue-with-traffic-4335-hd-ready (1)', 'mixkit-manhattan-avenue-with-traffic-4335-hd-ready', 'mixkit-person-walking-on-crosswalk-98-hd-ready (1)', 'mixkit-person-walking-on-crosswalk-98-hd-ready', 'mixkit-traffic-light-directing-traffic-4272-hd-ready (1)', 'mixkit-traffic-light-directing-traffic-4272-hd-ready']


## 5. Index: describe every segment with the VLM

In [24]:
def video_info(path):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    n = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
    cap.release()
    return fps, n / fps


def sample_frames(path, start, end, n=FRAMES_PER_SEGMENT):
    cap = cv2.VideoCapture(str(path))
    frames = []
    for i in range(n):
        t = start + (i + 0.5) * (end - start) / n
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        ok, frame = cap.read()
        if not ok:
            continue
        img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        scale = MAX_SIDE / max(img.size)
        if scale < 1:
            img = img.resize((int(img.width * scale), int(img.height * scale)))
        frames.append(img)
    cap.release()
    return frames


DESCRIBE_PROMPT = (
    "These are frames in time order from one short clip of a traffic or city camera. "
    "In two sentences, describe what happens: vehicle types and colors, pedestrians, "
    "cyclists, traffic signals, and any notable event such as a violation, near-miss, "
    "collision or blockage. Only state what you can see."
)

index = []  # one dict per segment
t0 = time.time()
for v in VIDEOS:
    fps, dur = video_info(v)
    s = 0.0
    while s < dur - 0.5:
        e = min(s + SEGMENT_SECONDS, dur)
        frames = sample_frames(v, s, e)
        if frames:
            desc = vlm(frames, DESCRIBE_PROMPT)
            index.append({
                "id": f"{v.stem}@{int(s)}",
                "camera": v.stem,
                "video": str(v),
                "start": s,
                "end": e,
                "description": desc,
            })
        s += SEGMENT_SECONDS
    print(f"Indexed {v.stem}: {sum(1 for r in index if r['camera'] == v.stem)} segments")

emb = embedder.encode([r["description"] for r in index], normalize_embeddings=True)
print(f"{len(index)} segments indexed in {time.time() - t0:.0f}s")
print(index[0]["id"], "->", index[0]["description"] if index else "")

Indexed mixkit-man-cycling-down-a-street-in-a-city-41878-hd-ready (1): 3 segments
Indexed mixkit-man-cycling-down-a-street-in-a-city-41878-hd-ready: 3 segments
Indexed mixkit-manhattan-avenue-with-traffic-4335-hd-ready (1): 2 segments
Indexed mixkit-manhattan-avenue-with-traffic-4335-hd-ready: 2 segments
Indexed mixkit-person-walking-on-crosswalk-98-hd-ready (1): 3 segments
Indexed mixkit-person-walking-on-crosswalk-98-hd-ready: 3 segments
Indexed mixkit-traffic-light-directing-traffic-4272-hd-ready (1): 6 segments
Indexed mixkit-traffic-light-directing-traffic-4272-hd-ready: 6 segments
28 segments indexed in 103s
mixkit-man-cycling-down-a-street-in-a-city-41878-hd-ready (1)@0 -> The video captures a cyclist riding a bicycle on a paved path in front of a building. The cyclist is wearing a beige shirt, black shorts, and a black cap. The bicycle is a standard road bike. The surroundings include greenery, a tree, and some bushes. There are no pedestrians or traffic signals visible in the 

## 6. Search and verify

Retrieval is by text embedding of the VLM descriptions. Each candidate is then **re-checked by the VLM against your exact question**. Weak candidates are dropped, and if nothing passes, the answer is "no verified match".

In [11]:
def fmt_t(sec):
    return f"{int(sec // 60):02d}:{int(sec % 60):02d}"


def search(query, camera=None, t_min=None, t_max=None, top_k=TOP_K, threshold=VERIFY_THRESHOLD):
    t0 = time.time()
    q = embedder.encode([query], normalize_embeddings=True)[0]
    sims = emb @ q
    order = np.argsort(-sims)
    cands = []
    for i in order:
        r = index[i]
        if camera and r["camera"] != camera:
            continue
        if t_min is not None and r["end"] < t_min:
            continue
        if t_max is not None and r["start"] > t_max:
            continue
        cands.append((i, float(sims[i])))
        if len(cands) >= top_k:
            break

    results = []
    for i, sim in cands:
        r = index[i]
        frames = sample_frames(r["video"], r["start"], r["end"])
        prompt = (
            f'Question: does this clip show the following? "{query}"\n'
            "Answer Yes or No first, then give one short reason based only on what is visible."
        )
        answer, p_yes = vlm(frames, prompt, max_new_tokens=60, yes_no=True)
        results.append({**r, "similarity": sim, "p_yes": p_yes, "verifier": answer})

    verified = sorted((x for x in results if x["p_yes"] >= threshold), key=lambda x: -x["p_yes"])
    print(f"Query: {query!r} | {len(cands)} candidates | {len(verified)} verified | {time.time() - t0:.1f}s")
    if not verified:
        print("No verified match. (Better to say so than to invent one.)")
    for x in verified:
        print(f"  [{x['id']}] {x['camera']} {fmt_t(x['start'])}-{fmt_t(x['end'])} "
              f"P(yes)={x['p_yes']:.2f}\n     {x['description']}")
    return verified, results


TEST_QUERIES = [
    "A yellow taxi on a busy avenue",           # cam1
    "Traffic stopped at a signal",              # cam2
    "A person crosses the street",              # cam3
    "A cyclist rides on the road",              # cam4
    "A car accident on a highway",              # cam5
    "A car runs a red light",                   # expect: no verified match
]

summary = {}
for q in TEST_QUERIES:
    ver, _ = search(q)
    summary[q] = [(x["camera"], round(x["p_yes"], 2)) for x in ver[:3]]
    print()

print("SUMMARY")
for q, hits in summary.items():
    print(f"{q!r:50} -> {hits if hits else 'no verified match'}")

Query: 'A yellow taxi on a busy avenue' | 8 candidates | 8 verified | 9.0s
  [mixkit-traffic-light-directing-traffic-4272-hd-ready@0] mixkit-traffic-light-directing-traffic-4272-hd-ready 00:00-00:05 P(yes)=0.99
     The video captures a bustling city street scene with various vehicles, including yellow taxis and cars. Pedestrians are seen crossing the street, and traffic signals are visible at the intersection. The scene includes a "All Traffic" sign, indicating a specific lane or direction for traffic. The video does not show any notable events or incidents.
  [mixkit-traffic-light-directing-traffic-4272-hd-ready@5] mixkit-traffic-light-directing-traffic-4272-hd-ready 00:05-00:10 P(yes)=0.99
     The video captures a busy street scene in New York City. A yellow taxi is driving down the street, while a pedestrian is crossing the street. There are also cars parked along the side of the road, and a traffic light is visible. The scene is bustling with activity, with people walking and veh

# 7. Export the case file

Builds a timeline report where every line points to a clip and timestamp. The narrative is assembled only from the VLM's descriptions of verified clips (no extra free-text generation), so it can't add claims that aren't in the footage. It also exports the clips, thumbnails and a JSON record.

In [25]:
def export_case(query, verified, case_name="case_001"):
    case_dir = OUT_DIR / case_name
    if case_dir.exists():
        shutil.rmtree(case_dir)
    (case_dir / "clips").mkdir(parents=True)
    (case_dir / "thumbs").mkdir()

    # Ensure query is a string for HTML/JSON formatting
    query_str = ", ".join(query) if isinstance(query, list) else str(query)

    rows = sorted(verified, key=lambda x: (x["camera"], x["start"]))
    items = []
    for x in rows:
        clip = case_dir / "clips" / f"{x['id'].replace('@', '_at_')}.mp4"
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-ss", str(x["start"]), "-to", str(x["end"]),
             "-i", x["video"], "-c:v", "libx264", "-preset", "veryfast", "-an", str(clip)],
            check=True,
        )
        thumb_img = sample_frames(x["video"], x["start"], x["end"], n=1)[0]
        thumb = case_dir / "thumbs" / (clip.stem + ".jpg")
        thumb_img.save(thumb, quality=85)
        items.append({**x, "clip_file": f"clips/{clip.name}", "thumb_file": f"thumbs/{thumb.name}"})

    parts = [
        "<html><head><meta charset='utf-8'><title>Case File</title>",
        "<style>body{font-family:sans-serif;max-width:900px;margin:2em auto}"
        ".item{border:1px solid #ccc;border-radius:8px;padding:12px;margin:12px 0;display:flex;gap:12px}"
        "img{width:240px;border-radius:4px}.meta{color:#555;font-size:13px}</style></head><body>",
        "<h1>Case File</h1>",
        f"<p><b>Query:</b> {html.escape(query_str)}</p>",
        f"<p class='meta'>Generated {time.strftime('%Y-%m-%d %H:%M')} | "
        f"{len(items)} verified segment(s) | threshold P(yes) &ge; {VERIFY_THRESHOLD}</p>",
        "<h2>Timeline</h2>",
    ]
    if not items:
        parts.append("<p><b>No verified match was found.</b></p>")
    for k, x in enumerate(items, 1):
        with open(case_dir / x["thumb_file"], "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
        parts.append(
            "<div class='item'>"
            f"<img src='data:image/jpeg;base64,{b64}'>"
            "<div>"
            f"<b>Evidence {k}: camera {html.escape(x['camera'])}, "
            f"{fmt_t(x['start'])}-{fmt_t(x['end'])}</b>"
            f"<div class='meta'>Segment {html.escape(x['id'])} | verifier confidence {x['p_yes']:.2f} | "
            f"<a href='{x['clip_file']}'>clip</a></div>"
            f"<p>{html.escape(x['description'])}</p>"
            f"<p class='meta'>Verifier note: {html.escape(x['verifier'])}</p>"
            "</div></div>"
        )
    parts.append(
        "<hr><p class='meta'>Automated analysis. Descriptions are model output and must be reviewed "
        "by a human before use as evidence.</p></body></html>"
    )
    (case_dir / "report.html").write_text("\n".join(parts), encoding="utf-8")
    (case_dir / "case.json").write_text(
        json.dumps({"query": query, "threshold": VERIFY_THRESHOLD, "items": items}, indent=2, default=float),
        encoding="utf-8",
    )

    zip_path = OUT_DIR / f"{case_name}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in case_dir.rglob("*"):
            if p.is_file():
                z.write(p, p.relative_to(case_dir))
    return zip_path

# Let's run search on a test query to populate the 'verified' list before exporting
verified, _ = search("A yellow taxi on a busy avenue")
zip_path = export_case("A yellow taxi on a busy avenue", verified)
print("Saved:", zip_path)
from google.colab import files as _f
_f.download(str(zip_path))

Query: 'A yellow taxi on a busy avenue' | 8 candidates | 8 verified | 8.3s
  [mixkit-traffic-light-directing-traffic-4272-hd-ready@5] mixkit-traffic-light-directing-traffic-4272-hd-ready 00:05-00:10 P(yes)=0.99
     The video captures a busy street scene in New York City. A yellow taxi is driving down the street, while a pedestrian is crossing the street. There are also cars parked along the side of the road, and a traffic light is visible. The scene is bustling with activity, with people walking and vehicles moving through the area.
  [mixkit-traffic-light-directing-traffic-4272-hd-ready (1)@5] mixkit-traffic-light-directing-traffic-4272-hd-ready (1) 00:05-00:10 P(yes)=0.99
     The video captures a busy street scene in New York City. A yellow taxi is driving down the street, while a pedestrian is crossing the street. There are also cars parked along the side of the road, and a traffic light is visible. The scene is bustling with activity, with people walking and vehicles moving throu

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Mini benchmark (use this in your pitch)

Fill `TESTS` with 20 queries where you know the right camera and time window. It reports precision, hit rate and time per query. Compare against how long it takes you to find the same incident by scrubbing manually.

In [26]:
# (query, camera, start_sec, end_sec). camera=None means: no match is expected.
TESTS = [
    # cam1: busy avenue (vehicles)
    ("A yellow taxi on a busy avenue", "cam1", 0, 9999),
    ("Many cars driving on a city avenue", "cam1", 0, 9999),
    ("A bus on the street", "cam1", 0, 9999),
    ("Tall buildings on both sides of a busy road", "cam1", 0, 9999),
    # cam2: traffic light / junction
    ("Traffic stopped at a signal", "cam2", 0, 9999),
    ("A traffic light visible above the road", "cam2", 0, 9999),
    ("Cars waiting at a junction", "cam2", 0, 9999),
    ("Vehicles moving through a junction", "cam2", 0, 9999),
    # cam3: pedestrian crosswalk
    ("A person crosses the street", "cam3", 0, 9999),
    ("A pedestrian walks on a crosswalk", "cam3", 0, 9999),
    ("A person walking on the road", "cam3", 0, 9999),
    ("A pedestrian crossing in front of a car", "cam3", 0, 9999),
    # cam4: cyclist
    ("A cyclist rides on the road", "cam4", 0, 9999),
    ("A man cycling down a city street", "cam4", 0, 9999),
    ("A bicycle next to parked cars", "cam4", 0, 9999),
    # cam5: accident
    ("A car accident on a highway", "cam5", 0, 9999),
    ("A crashed vehicle on the road", "cam5", 0, 9999),
    ("Damaged vehicles blocking a lane", "cam5", 0, 9999),
    # No match expected (tests that the verifier does not invent results)
    ("A car runs a red light", None, 0, 0),
    ("A cyclist collides with a car", None, 0, 0),
]


import csv
from IPython.display import HTML, display

LOOSE_WINDOW_SECONDS = 100  # a window this long or longer is flagged as "loose" in the report


def pct(a, b):
    return f"{a / b:.0%}" if b else "n/a"


def run_benchmark(tests):
    rows = []
    for q, cam, gs, ge in tests:
        t0 = time.time()
        ver, res = search(q)
        dt = time.time() - t0
        negative = cam is None
        if negative:
            ok = not ver
            status = "PASS" if ok else "FAIL"
            n_correct = 0
        else:
            match = [x for x in ver if x["camera"] == cam and x["start"] < ge and x["end"] > gs]
            ok = bool(match)
            status = "HIT" if ok else "MISS"
            n_correct = len(match)
        best = max(res, key=lambda r: r["p_yes"]) if res else None
        rows.append({
            "query": q,
            "expected": "(no match)" if negative else cam,
            "window": "-" if negative else f"{fmt_t(gs)}-{fmt_t(ge) if ge < 9999 else 'end'}",
            "status": status,
            "ok": ok,
            "negative": negative,
            "loose": (not negative) and (ge - gs) >= LOOSE_WINDOW_SECONDS,
            "n_verified": len(ver),
            "n_correct": n_correct,
            "seconds": dt,
            "top": f"{best['camera']} {fmt_t(best['start'])}-{fmt_t(best['end'])}" if best else "-",
            "top_p": best["p_yes"] if best else 0.0,
            "top_desc": best["description"] if best else "",
        })
    return rows


def summarize(rows):
    pos = [r for r in rows if not r["negative"]]
    neg = [r for r in rows if r["negative"]]
    returned = sum(r["n_verified"] for r in rows)
    correct = sum(r["n_correct"] for r in pos)
    cams = {}
    for r in pos:
        c = cams.setdefault(r["expected"], [0, 0])
        c[1] += 1
        c[0] += 1 if r["ok"] else 0
    return {
        "pos_total": len(pos),
        "pos_hits": sum(1 for r in pos if r["ok"]),
        "neg_total": len(neg),
        "neg_pass": sum(1 for r in neg if r["ok"]),
        "returned": returned,
        "correct": correct,
        "avg_seconds": sum(r["seconds"] for r in rows) / max(len(rows), 1),
        "cams": cams,
        "any_loose": any(r["loose"] for r in pos),
    }


def build_benchmark_report(rows, s):
    esc = html.escape
    colors = {"HIT": "#1a7f37", "PASS": "#1a7f37", "MISS": "#b42318", "FAIL": "#b42318"}
    headline = (
        f"A correct segment was found for {s['pos_hits']} of {s['pos_total']} queries "
        f"({pct(s['pos_hits'], s['pos_total'])}). "
        f"{s['neg_pass']} of {s['neg_total']} queries with no true match correctly returned no result. "
        f"Average time per query: {s['avg_seconds']:.1f}s."
    )
    p = [
        "<html><head><meta charset='utf-8'><title>Case File benchmark</title>",
        "<style>body{font-family:sans-serif;max-width:980px;margin:2em auto;padding:0 12px}"
        "table{border-collapse:collapse;width:100%;margin:12px 0}"
        "th,td{border:1px solid #ddd;padding:6px 8px;text-align:left;font-size:13px;vertical-align:top}"
        "th{background:#f4f4f4}.big{font-size:28px;font-weight:700}"
        ".cards{display:flex;gap:12px;flex-wrap:wrap}"
        ".card{border:1px solid #ddd;border-radius:8px;padding:12px 16px;min-width:170px}"
        ".meta{color:#555;font-size:13px}"
        ".warn{background:#fff7e6;border:1px solid #f0c36d;padding:8px 12px;border-radius:6px}</style></head><body>",
        "<h1>Case File: benchmark report</h1>",
        f"<p class='meta'>Generated {time.strftime('%Y-%m-%d %H:%M')} | model {esc(MODEL_ID)} | "
        f"{len(index)} indexed segments | {len(VIDEOS)} camera(s) | "
        f"segment {SEGMENT_SECONDS}s, {FRAMES_PER_SEGMENT} frames | top-k {TOP_K} | "
        f"verify threshold P(yes) &ge; {VERIFY_THRESHOLD}</p>",
        f"<p><b>{esc(headline)}</b></p>",
        "<div class='cards'>",
        f"<div class='card'><div class='meta'>Hit rate</div><div class='big'>{pct(s['pos_hits'], s['pos_total'])}</div>"
        f"<div class='meta'>{s['pos_hits']} / {s['pos_total']} queries</div></div>",
        f"<div class='card'><div class='meta'>Precision</div><div class='big'>{pct(s['correct'], s['returned'])}</div>"
        f"<div class='meta'>{s['correct']} correct / {s['returned']} verified results</div></div>",
        f"<div class='card'><div class='meta'>No-match queries</div><div class='big'>{pct(s['neg_pass'], s['neg_total'])}</div>"
        f"<div class='meta'>{s['neg_pass']} / {s['neg_total']} correctly empty</div></div>",
        f"<div class='card'><div class='meta'>Avg time / query</div><div class='big'>{s['avg_seconds']:.1f}s</div></div>",
        "</div>",
    ]
    if s["any_loose"]:
        p.append(
            "<p class='warn'><b>Caution:</b> some ground-truth windows are loose (the whole clip). "
            "Hit rate and precision are therefore optimistic. Tighten the windows in TESTS for a stricter result.</p>"
        )
    p.append("<h2>By camera</h2><table><tr><th>Camera</th><th>Hits</th><th>Queries</th><th>Hit rate</th></tr>")
    for cam, (h, t) in sorted(s["cams"].items()):
        p.append(f"<tr><td>{esc(cam)}</td><td>{h}</td><td>{t}</td><td>{pct(h, t)}</td></tr>")
    p.append("</table>")
    p.append(
        "<h2>All queries</h2><table><tr><th>Status</th><th>Query</th><th>Expected</th><th>Window</th>"
        "<th>Verified</th><th>Best candidate (P yes)</th><th>Seconds</th></tr>"
    )
    for r in rows:
        p.append(
            f"<tr><td style='color:{colors[r['status']]};font-weight:700'>{r['status']}</td>"
            f"<td>{esc(r['query'])}</td><td>{esc(r['expected'])}</td><td>{esc(r['window'])}</td>"
            f"<td>{r['n_verified']}</td><td>{esc(r['top'])} ({r['top_p']:.2f})</td><td>{r['seconds']:.1f}</td></tr>"
        )
    p.append("</table>")
    bad = [r for r in rows if not r["ok"]]
    p.append("<h2>Misses and failures</h2>")
    if not bad:
        p.append("<p>None.</p>")
    for r in bad:
        p.append(
            f"<div style='margin:8px 0'><b style='color:{colors[r['status']]}'>{r['status']}</b> "
            f"{esc(r['query'])} (expected: {esc(r['expected'])})"
            f"<div class='meta'>Closest candidate {esc(r['top'])}, P(yes) {r['top_p']:.2f}: {esc(r['top_desc'])}</div></div>"
        )
    p.append(
        "<h2>Notes</h2><ul>"
        "<li>Small test set: treat the numbers as indicative, not as a general accuracy claim.</li>"
        "<li>Precision counts every verified result for a query, so returning extra segments lowers it.</li>"
        "<li>Descriptions and verdicts are model output and need human review before use as evidence.</li>"
        "</ul></body></html>"
    )
    return "".join(p)


if not TESTS:
    print("Add test cases to TESTS first.")
else:
    rows = run_benchmark(TESTS)
    s = summarize(rows)
    report_path = OUT_DIR / "benchmark_report.html"
    csv_path = OUT_DIR / "benchmark_results.csv"
    report_path.write_text(build_benchmark_report(rows, s), encoding="utf-8")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print("---")
    print(f"Hit rate: {s['pos_hits']}/{s['pos_total']} = {pct(s['pos_hits'], s['pos_total'])}")
    print(f"Precision: {s['correct']}/{s['returned']} = {pct(s['correct'], s['returned'])}")
    print(f"No-match queries correctly empty: {s['neg_pass']}/{s['neg_total']}")
    print(f"Avg time per query: {s['avg_seconds']:.1f}s")
    print("Saved:", report_path, "and", csv_path)
    display(HTML(report_path.read_text(encoding="utf-8")))
    try:
        from google.colab import files
        files.download(str(report_path))
        files.download(str(csv_path))
    except Exception as e:
        print("Auto-download did not start. Get both files from the Files panel (casefile > out):", e)

Query: 'A yellow taxi on a busy avenue' | 8 candidates | 8 verified | 8.5s
  [mixkit-traffic-light-directing-traffic-4272-hd-ready@5] mixkit-traffic-light-directing-traffic-4272-hd-ready 00:05-00:10 P(yes)=0.99
     The video captures a busy street scene in New York City. A yellow taxi is driving down the street, while a pedestrian is crossing the street. There are also cars parked along the side of the road, and a traffic light is visible. The scene is bustling with activity, with people walking and vehicles moving through the area.
  [mixkit-traffic-light-directing-traffic-4272-hd-ready (1)@5] mixkit-traffic-light-directing-traffic-4272-hd-ready (1) 00:05-00:10 P(yes)=0.99
     The video captures a busy street scene in New York City. A yellow taxi is driving down the street, while a pedestrian is crossing the street. There are also cars parked along the side of the road, and a traffic light is visible. The scene is bustling with activity, with people walking and vehicles moving throu

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>